# 🎓 MentorX — StarCoder2-7B SFT (QLoRA)

**Model:** StarCoder2-7B-Instruct  
**Yöntem:** QLoRA (4-bit) via Unsloth  
**Veri:** ~765 Türkçe Sokratik diyalog (Python CS1)  
**Hedef:** MentorX Python CS1 Sokratik tutor davranışını öğretmek  

> ⚠️ **Colab Pro / A100 (40GB) önerilir.**

## 📦 1. Kurulum

In [1]:
%%capture
!pip install -U unsloth wandb scikit-learn

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # CUDA hatalarını senkron göster

import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 🔧 2. Konfigürasyon

In [ ]:
# Model
MODEL_NAME       = "bigcode/starcoder2-7b"
MAX_SEQ_LENGTH   = 1024
DTYPE            = None
LOAD_IN_4BIT     = True

# LoRA
LORA_R           = 8
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.0
TARGET_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj", "c_proj", "c_fc"]

# Eğitim
OUTPUT_DIR       = "./mentorx-starcoder2-7b-lora"
NUM_EPOCHS       = 5
BATCH_SIZE       = 4
GRAD_ACCUM       = 4
LEARNING_RATE    = 1e-4
WARMUP_STEPS     = 30
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0
LOGGING_STEPS    = 10
SAVE_STEPS       = 20
EVAL_STEPS       = 20
SEED             = 42

# Veri
INPUT_FILE       = "train_py_cs1.jsonl"
TRAIN_FILE       = "train_py_cs1_train.jsonl"
VAL_FILE         = "train_py_cs1_val.jsonl"
VAL_RATIO        = 0.15

# HuggingFace Hub
HF_PUSH          = True
HF_REPO          = "mtepe01/mentorx-starcoder2-7b"

# WandB
USE_WANDB        = False
WANDB_PROJECT    = "mentorx-sft-python"

print("✅ Konfigürasyon yüklendi")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

✅ Konfigürasyon yüklendi
   Effective batch size: 16


## 📂 3. Veri Yükleme & Stratified Train/Val Split

In [ ]:
from google.colab import files

print("train_py_cs1.jsonl dosyasını seç:")
uploaded = files.upload()

for fname in uploaded:
    print(f"✅ Yüklendi: {fname} ({len(uploaded[fname])/1024:.1f} KB)")

train_py_cs1.jsonl dosyasını seç:


Saving train_py_cs1.jsonl to train_py_cs1.jsonl
✅ Yüklendi: train_py_cs1.jsonl (1976.9 KB)


In [ ]:
import json
from collections import Counter
from sklearn.model_selection import train_test_split

with open(INPUT_FILE, encoding="utf-8") as f:
    all_records = [json.loads(line) for line in f if line.strip()]

print(f"Toplam diyalog: {len(all_records)}")

profiles = [r.get("profile", "UNKNOWN") for r in all_records]
print(f"Profil dağılımı: {Counter(profiles)}")

train_records, val_records = train_test_split(
    all_records,
    test_size    = VAL_RATIO,
    random_state = SEED,
    stratify     = profiles,
)

print(f"\nTrain: {len(train_records)} | Val: {len(val_records)}")
print(f"Train profil: {Counter(r.get('profile', 'UNKNOWN') for r in train_records)}")
print(f"Val profil  : {Counter(r.get('profile', 'UNKNOWN') for r in val_records)}")

with open(TRAIN_FILE, "w", encoding="utf-8") as f:
    for r in train_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(VAL_FILE, "w", encoding="utf-8") as f:
    for r in val_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\n✅ {TRAIN_FILE} ve {VAL_FILE} oluşturuldu")

Toplam diyalog: 765
Profil dağılımı: Counter({'UNKNOWN': 765})

Train: 650 | Val: 115
Train profil: Counter({'UNKNOWN': 650})
Val profil  : Counter({'UNKNOWN': 115})

✅ train_py_cs1_train.jsonl ve train_py_cs1_val.jsonl oluşturuldu


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE}
)

print(f"Train örnekleri : {len(dataset['train'])}")
print(f"Val örnekleri   : {len(dataset['validation'])}")

sample = dataset["train"][0]
print(f"\n--- Örnek diyalog ({len(sample['conversations'])} mesaj) ---")
for msg in sample["conversations"][:3]:
    role = msg["from"].upper()
    preview = msg["value"][:80] + "..." if len(msg["value"]) > 80 else msg["value"]
    print(f"[{role}]: {preview}")

def validate_split(ds, name):
    for i, row in enumerate(ds):
        convs = row.get("conversations")
        assert isinstance(convs, list) and len(convs) >= 2, f"{name}[{i}] invalid"
        assert convs[-1].get("from") == "gpt", f"{name}[{i}] last must be gpt"
        for j, m in enumerate(convs):
            assert m.get("from") in {"system", "human", "gpt"}, f"{name}[{i}] bad role @{j}"
            v = m.get("value")
            assert isinstance(v, str) and v.strip(), f"{name}[{i}] empty @{j}"

validate_split(dataset["train"],      "train")
validate_split(dataset["validation"], "validation")
print("✅ Dataset şeması doğrulandı")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train örnekleri : 650
Val örnekleri   : 115

--- Örnek diyalog (9 mesaj) ---
[SYSTEM]: Sen bir Python programlama tutor'ısın. Öğrenciye Sokratik yöntemle rehberlik ede...
[HUMAN]: Kod çalışıyor ama çıktı [1, 2] veriyor, beklenen [1, 2, 3]. TypeError yok, yani ...
[GPT]: Güzel bir gözlem. Sence `liste + [3]` ifadesi `liste` değişkeninin kendisini değ...
✅ Dataset şeması doğrulandı


## 🤖 4. Model Yükleme & LoRA Modül Doğrulama

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
)

print(f"✅ Model yüklendi: {MODEL_NAME}")
print(f"   Model vocab size : {model.config.vocab_size}")
print(f"   Tokenizer vocab  : {len(tokenizer)}")
print(f"   Parametre sayısı : {model.num_parameters()/1e9:.2f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Starcoder2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/41.6k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

bigcode/starcoder2-7b does not have a padding token! Will use pad_token = <|endoftext|>.
✅ Model yüklendi: bigcode/starcoder2-7b
   Model vocab size : 49152
   Tokenizer vocab  : 49152
   Parametre sayısı : 7.17B


In [ ]:
# StarCoder2 katman adlarını doğrula
linear_names = set()
for name, module in model.named_modules():
    if hasattr(module, 'weight') and 'Linear' in type(module).__name__:
        short = name.split('.')[-1]
        linear_names.add(short)

print("Modeldeki linear katman adları:")
for n in sorted(linear_names):
    print(f"  {n}")

missing = [m for m in TARGET_MODULES if m not in linear_names]
if missing:
    raise ValueError(
        f"TARGET_MODULES modelde yok: {missing}\n"
        f"Yukarıdaki listeden gerçek isimlerle güncelle."
    )

print(f"\n✅ Tüm TARGET_MODULES modelde mevcut: {TARGET_MODULES}")

Modeldeki linear katman adları:
  c_fc
  c_proj
  k_proj
  lm_head
  o_proj
  q_proj
  v_proj

✅ Tüm TARGET_MODULES modelde mevcut: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'c_proj', 'c_fc']


## 🔗 5. LoRA Adapter Ekleme

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
)

print("✅ LoRA adapter eklendi")
model.print_trainable_parameters()

✅ LoRA adapter eklendi
trainable params: 19,136,512 || all params: 7,193,060,352 || trainable%: 0.2660


## 🔄 6. Veri Formatlama

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    map_eos_token = True,
)

# Vocab boyutu kontrol — yeni token eklendiyse embedding'i eşitle
if len(tokenizer) != model.config.vocab_size:
    print(f"⚠️  Vocab uyumsuzluğu: tokenizer={len(tokenizer)}, model={model.config.vocab_size}")
    print("   Embedding resize yapılıyor...")
    model.resize_token_embeddings(len(tokenizer))
    print(f"✅ Embedding yeniden boyutlandırıldı: {len(tokenizer)}")
else:
    print(f"✅ Vocab boyutu eşleşiyor: {len(tokenizer)}")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = []
    for convo in convos:
        messages = []
        for m in convo:
            role_map = {"system": "system", "human": "user", "gpt": "assistant"}
            messages.append({
                "role": role_map[m["from"]],
                "content": m["value"]
            })
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

print("✅ Veri formatlandı")
print("\n--- Örnek formatted text (ilk 300 karakter) ---")
print(dataset["train"][0]["text"][:300])

Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model does not have a padding token! Will use pad_token = <|endoftext|>.
⚠️  Vocab uyumsuzluğu: tokenizer=49153, model=49152
   Embedding resize yapılıyor...
✅ Embedding yeniden boyutlandırıldı: 49153


Map:   0%|          | 0/650 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

✅ Veri formatlandı

--- Örnek formatted text (ilk 300 karakter) ---
<|im_start|>system
Sen bir Python programlama tutor'ısın. Öğrenciye Sokratik yöntemle rehberlik edersin. Tüm diyalog Türkçe olmalı — TypeError, for, range gibi teknik terimler ve kod keyword'leri hariç.

Kesinlikle yapmaman gerekenler:
- Kodu doğrudan yazma veya tamamlama
- Hatayı doğrudan gösterme



## 🏋️ 7. Eğitim

In [ ]:
import os

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name="starcoder2-7b-python-cs1")
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("WandB devre dışı")

WandB devre dışı


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset["train"],
    eval_dataset       = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size  = BATCH_SIZE,
        per_device_eval_batch_size   = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        num_train_epochs            = NUM_EPOCHS,
        learning_rate               = LEARNING_RATE,
        max_grad_norm               = MAX_GRAD_NORM,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = LOGGING_STEPS,
        save_steps                  = SAVE_STEPS,
        eval_strategy               = "steps",
        eval_steps                  = EVAL_STEPS,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        optim                       = "adamw_8bit",
        weight_decay                = WEIGHT_DECAY,
        lr_scheduler_type           = LR_SCHEDULER,
        seed                        = SEED,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "wandb" if USE_WANDB else "none",
    ),
)

print("✅ Trainer hazır, eğitim başlıyor...")
trainer_stats = trainer.train()

print(f"\n✅ Eğitim tamamlandı!")
print(f"   Toplam adım    : {trainer_stats.global_step}")
print(f"   Eğitim süresi  : {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"   Son train loss : {trainer_stats.metrics['train_loss']:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/650 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/115 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 49152, 'pad_token_id': 49152}.


✅ Trainer hazır, eğitim başlıyor...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 650 | Num Epochs = 5 | Total steps = 205
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 19,136,512 of 7,193,064,960 (0.27% trained)


Step,Training Loss,Validation Loss
20,2.658741,1.661358
40,1.594266,0.999286
60,1.086989,0.802656
80,0.957135,0.752082
100,0.892070,0.721163
120,0.873869,0.702098
140,0.840198,0.693206
160,0.837467,0.686360
180,0.794463,0.684463
200,0.808377,0.683914


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in ./mentorx-starcoder2-7b-lora/checkpoint-20/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in ./mentorx-starcoder2-7b-lora/checkpoint-40/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in ./mentorx-starcoder2-7b-lora/checkpoint-60/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packag


✅ Eğitim tamamlandı!
   Toplam adım    : 205
   Eğitim süresi  : 1276.1s
   Son train loss : 1.1832


## 💾 8. Model Kaydetme

In [16]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA ağırlıkları kaydedildi: {OUTPUT_DIR}")

✅ LoRA ağırlıkları kaydedildi: ./mentorx-starcoder2-7b-lora


In [17]:
if HF_PUSH:
    from huggingface_hub import login
    login()
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f"✅ HF Hub'a yüklendi: https://huggingface.co/{HF_REPO}")
else:
    print("HF_PUSH=False — Hub'a yükleme atlandı")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  20%|#9        |  192MB /  983MB            

No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpl1hagwq3/tokenizer_config.json.


Saved model to https://huggingface.co/mtepe01/mentorx-starcoder2-7b


No files have been modified since last commit. Skipping to prevent empty commit.


✅ HF Hub'a yüklendi: https://huggingface.co/mtepe01/mentorx-starcoder2-7b


In [18]:
import shutil, os
from google.colab import files

zip_path = "mentorx_starcoder2_lora_weights"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
zip_size = os.path.getsize(f"{zip_path}.zip") / 1024 / 1024
print(f"✅ ZIP oluşturuldu: {zip_path}.zip ({zip_size:.1f} MB)")
files.download(f"{zip_path}.zip")

✅ ZIP oluşturuldu: mentorx_starcoder2_lora_weights.zip (9368.2 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🧪 9. Davranış Testleri (Inference)

In [24]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "Sen bir Python programlama tutor'ısın. Öğrenciye Sokratik yöntemle rehberlik edersin. "
    "Öğrenciye asla doğrudan cevap vermezsin, yönlendirici sorular sorarsın, max 3 cümleyle konuşursun."
)

def ask(user_message, history=None, max_new_tokens=200):
    if history is None:
        history = [{"role": "system", "content": SYSTEM_PROMPT}]
    history.append({"role": "user", "content": user_message})
    inputs = tokenizer.apply_chat_template(
        history,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids          = inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    history.append({"role": "assistant", "content": response})
    return response, history

print("✅ Inference fonksiyonu hazır")

✅ Inference fonksiyonu hazır


In [20]:
print("=" * 60)
print("TEST 1: Sokratik davranış")
print("=" * 60)
resp, _ = ask("Liste ve tuple arasındaki fark nedir?")
print(f"👤 Öğrenci: Liste ve tuple arasındaki fark nedir?")
print(f"🎓 MentorX: {resp}\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 1: Sokratik davranış
👤 Öğrenci: Liste ve tuple arasındaki fark nedir?
🎓 MentorX: İki türün ortak özellikleri var mı sence? Yoksa tek bir liste içindeki her elemanın türü ne olabilir?



In [21]:
print("=" * 60)
print("TEST 2: Direkt cevap isteği")
print("=" * 60)
resp, _ = ask("Bana cevabı direkt söyle, vakit kaybetmek istemiyorum.")
print(f"👤 Öğrenci: Bana cevabı direkt söyle, vakit kaybetmek istemiyorum.")
print(f"🎓 MentorX: {resp}\n")

Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 2: Direkt cevap isteği
👤 Öğrenci: Bana cevabı direkt söyle, vakit kaybetmek istemiyorum.
🎓 MentorX: Evet, kodunu yazdığında hata mesajını aldığında ne anlamındasın? Hatanın hangi satırda olduğunu ve ne yapıyorduklarına bakalım.



In [22]:
print("=" * 60)
print("TEST 3: Onay isteği")
print("=" * 60)
resp, _ = ask("Cevabı zaten biliyorum, sadece doğrula: liste mutable, tuple immutable.")
print(f"👤 Öğrenci: Cevabı zaten biliyorum, sadece doğrula: liste mutable, tuple immutable.")
print(f"🎓 MentorX: {resp}\n")

Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 3: Onay isteği
👤 Öğrenci: Cevabı zaten biliyorum, sadece doğrula: liste mutable, tuple immutable.
🎓 MentorX: Güzel bir başlangıç. Peki, `liste[0] = 'me'` ifadesini yazdığında ne olur? Yani listenin ilk elemanını nasıl değiştirebilirsin?



In [23]:
print("=" * 60)
print("TEST 4: Çok turlu diyalog")
print("=" * 60)

history = None
turns = [
    "Mutable kavramını anlamadım.",
    "Değişebilir demek sanırım?",
    "Yani liste içindeki elemanı değiştirebilirim ama tuple'ı değiştiremem.",
]

for q in turns:
    resp, history = ask(q, history)
    print(f"👤 Öğrenci: {q}")
    print(f"🎓 MentorX: {resp}")
    print("-" * 40)

Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 4: Çok turlu diyalog


Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


👤 Öğrenci: Mutable kavramını anlamadım.
🎓 MentorX: Mutable kelimesi, değişkene atama yapıldığında ne olur? Demek ki bir değişkeni tanımlarken hangi türde bir değer verebilirsin?
----------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


👤 Öğrenci: Değişebilir demek sanırım?
🎓 MentorX: Doğru yoldasın. Peki, `a = 5` ifadesinde `a`'nin değerinin 5 olması için hangi türde bir değer kullanabilirsin?
----------------------------------------
👤 Öğrenci: Yani liste içindeki elemanı değiştirebilirim ama tuple'ı değiştiremem.
🎓 MentorX: Harika bir açıklama! Şimdi şunu düşün: Bir listedeki her bir elemana erişmek için hangi işlemi yapabilirsin? Yani `liste[0]` gibi. Bu işlemi uygulamak için `liste` değişkenini nasıl tanımlamalısın?
----------------------------------------
